# Merge com dados AFYA
Agrega `access_afya_dengue_2021_2026.csv` (diário × município) para semanal × UF e une com `dengue_uf_climate.csv`.

In [1]:
import pandas as pd
import numpy as np

## 1. Carregamento

In [2]:
dengue = pd.read_csv('dengue_uf_climate.csv', parse_dates=['date'])

# AFYA é grande (~9M linhas) — carrega com dtypes enxutos
afya = pd.read_csv(
    '../DataRaw/access_afya_dengue_2021_2026.csv',
    parse_dates=['access_date'],
    dtype={'geocode': 'int32', 'access_count': 'float32'}
)

print('dengue:', dengue.shape)
print('afya:  ', afya.shape)
afya.head()

dengue: (21970, 49)
afya:   (8963730, 5)


,access_date,geocode,uf,accessed_disease,access_count
0,2021-01-01,2927408,BA,Dengue,23.0
1,2021-01-01,5108402,MT,Dengue,0.0
2,2021-01-01,2304400,CE,Dengue,0.0
3,2021-01-01,4205407,SC,Dengue,0.0
4,2021-01-01,1503606,PA,Dengue,0.0


## 2. Exclusão do Espírito Santo

In [3]:
afya = afya[afya['uf'] != 'ES'].copy()
print('Após exclusão ES:', afya.shape)

Após exclusão ES: (8817432, 5)


## 3. Alinhamento à semana epidemiológica
No Brasil, a semana epidemiológica começa no domingo. Para cada data diária, encontramos o domingo anterior (início da semana).

In [4]:
# dayofweek: segunda=0 … domingo=6
# dias desde o último domingo: (dayofweek + 1) % 7
days_since_sunday = (afya['access_date'].dt.dayofweek + 1) % 7
afya['epiweek_start'] = afya['access_date'] - pd.to_timedelta(days_since_sunday, unit='D')

# Verificação
print('Dias da semana de epiweek_start (deve ser apenas 6 = domingo):')
print(afya['epiweek_start'].dt.dayofweek.value_counts())

Dias da semana de epiweek_start (deve ser apenas 6 = domingo):
6    8817432
Name: epiweek_start, dtype: int64


## 4. Agregação para semanal × UF

In [5]:
afya_uf = (
    afya
    .groupby(['epiweek_start', 'uf'], as_index=False)
    .agg(afya_access_count=('access_count', 'sum'))
    .rename(columns={'epiweek_start': 'date'})
)

print('afya_uf:', afya_uf.shape)
print('Periodo AFYA:', afya_uf['date'].min(), 'a', afya_uf['date'].max())
afya_uf.head()

afya_uf: (7384, 3)
Periodo AFYA: 2020-12-27 00:00:00 a 2026-05-31 00:00:00


,date,uf,afya_access_count
0,2020-12-27,AC,0.0
1,2020-12-27,AL,0.0
2,2020-12-27,AM,18.0
3,2020-12-27,AP,0.0
4,2020-12-27,BA,23.0


## 5. Merge com dengue_uf_climate

In [6]:
df = dengue.merge(afya_uf, on=['date', 'uf'], how='left')

total = len(df)
com_afya = df['afya_access_count'].notna().sum()
print(f'Linhas com dados AFYA: {com_afya:,} de {total:,} ({com_afya/total*100:.1f}%)')
print('(AFYA cobre 2021-2026; dados anteriores ficam como NaN)')
print('\nShape final:', df.shape)

Linhas com dados AFYA: 7,072 de 21,970 (32.2%)
(AFYA cobre 2021-2026; dados anteriores ficam como NaN)

Shape final: (21970, 50)


## 6. Verificação final

In [8]:
print('Colunas finais:')
print(df.dtypes.to_string())
df

Colunas finais:
date                 datetime64[ns]
epiweek                       int64
year                          int64
uf                           object
uf_code                       int64
casos                         int64
population                  float64
casos_norm                  float64
train_1                        bool
target_1                       bool
train_2                        bool
target_2                       bool
train_3                        bool
target_3                       bool
train_4                        bool
target_4                       bool
temp_min                    float64
temp_med                    float64
temp_max                    float64
precip_min                  float64
precip_med                  float64
precip_max                  float64
pressure_min                float64
pressure_med                float64
pressure_max                float64
rel_humid_min               float64
rel_humid_med               float64
rel_humid_ma

,date,epiweek,year,uf,uf_code,casos,population,casos_norm,train_1,target_1,...,temp_med_4m,temp_med_5m,temp_med_6m,umid_med_1m,umid_med_2m,umid_med_3m,umid_med_4m,umid_med_5m,umid_med_6m,afya_access_count
0,2010-01-03,201001,2010,AC,12,760,765021.0,99.343678,True,False,...,24.853806,24.180682,23.612581,88.156356,88.805525,89.132780,88.593928,87.774024,84.182341,NaN
1,2010-01-03,201001,2010,AL,27,73,3131625.0,2.331058,True,False,...,26.004582,24.943272,23.890278,70.867160,73.657448,75.478745,78.997803,79.573376,78.184609,NaN
2,2010-01-03,201001,2010,AM,13,155,3551196.0,4.364727,True,False,...,25.937014,25.661988,25.622033,87.019295,86.906537,87.576789,89.443343,88.733894,86.201776,NaN
3,2010-01-03,201001,2010,AP,16,34,686987.0,4.949148,True,False,...,25.393942,25.274962,25.566757,88.364605,85.868128,85.417012,88.207119,89.682035,87.066366,NaN
4,2010-01-03,201001,2010,BA,29,539,14187523.0,3.799113,True,False,...,24.381107,23.332367,22.199959,70.468094,73.472799,77.197583,77.668033,74.340060,72.315234,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21965,2026-03-08,202610,2026,RS,43,204,0.0,inf,False,False,...,14.114238,13.932773,15.358003,75.014146,78.041771,80.791930,81.303614,79.928919,78.246955,69.0
21966,2026-03-08,202610,2026,SC,42,354,0.0,inf,False,False,...,14.424885,14.174795,15.433945,79.231512,80.873468,82.252418,82.271608,81.266505,79.660438,52.0
21967,2026-03-08,202610,2026,SE,28,7,0.0,inf,False,False,...,24.134864,23.407584,23.540579,76.619938,78.689583,80.449928,79.610966,78.839348,75.633750,23.0
21968,2026-03-08,202610,2026,SP,35,2017,0.0,inf,False,False,...,18.905730,18.804678,20.642635,76.140858,75.296242,71.173927,68.979320,65.149430,59.505074,262.0


## 7. Exportação

In [9]:
df.to_csv('dengue_uf_final.csv', index=False)
print('Salvo em: 1_Preprocessing/dengue_uf_final.csv')

Salvo em: 1_Preprocessing/dengue_uf_final.csv
